# Day 3 — Classical Text Classification from Scratch

This notebook turns `docs/day3/theory.md` into working code. We build the whole classical
pipeline — **bag of words → TF-IDF → naive Bayes / logistic regression → precision, recall, F1** —
and then run it on 50,000 IMDb movie reviews.

**How to work through this notebook:**

1. Run cells top to bottom.
2. Six functions are left for **you** to implement — marked `TODO 1` … `TODO 6`.
3. After each TODO there is a **check cell**. The checks use the *review corpus* — the same
   six one-line reviews from the theory doc — so every expected number is one we already
   worked out by hand. If the check prints ✓, your code is right.
4. The real-data half (from "Part 2" onward) uses scikit-learn, so it runs whether or not
   you've finished the TODOs. Your from-scratch code gets validated against sklearn along the way.

| Step | Builds | Theory section |
|---|---|---|
| 1 | **TODO 1**: bag of words | 4.1 |
| 2 | **TODO 2**: IDF = Shannon surprise | 4.4 |
| 3 | **TODO 3**: TF-IDF + L2 normalization | 4.5–4.6 |
| 4 | **TODO 4**: multinomial naive Bayes | 5.4–5.5 |
| 5 | **TODO 5**: logistic regression + gradient descent | 6.1–6.3 |
| 6 | **TODO 6**: precision / recall / F1 | 8.3–8.4 |
| 7 | IMDb: the real experiment | 7 |
| 8 | Regularization, learning curves, failure modes | 6.4, 6.5, 9 |

In [ ]:
%pip install -r requirements.txt

In [ ]:
import math
import re
from collections import Counter

import numpy as np

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)

## The review corpus — our unit tests

Six one-line movie reviews, three positive and three negative:

```
POSITIVE                    NEGATIVE
D1: great film              D4: bad film
D2: great great story       D5: bad bad story
D3: good film               D6: dull film
```

It is deliberately symmetric (`great` mirrors `bad`, `good` mirrors `dull`), so when two
classifiers disagree we know the disagreement comes from the *method*, not from lopsided data.

Every number your code produces below, we already computed by hand in the theory doc.

In [ ]:
REVIEWS = [
    "great film",         # D1  positive
    "great great story",  # D2  positive
    "good film",          # D3  positive
    "bad film",           # D4  negative
    "bad bad story",      # D5  negative
    "dull film",          # D6  negative
]
LABELS = np.array([1, 1, 1, 0, 0, 0])   # 1 = positive, 0 = negative

def tokenize(text):
    '''Lowercase and pull out word-ish runs of characters.'''
    return re.findall(r"[a-z0-9']+", text.lower())

for r, y in zip(REVIEWS, LABELS):
    print(f"{'pos' if y else 'neg'}  {tokenize(r)}")

## Step 1 — TODO 1: bag of words

Theory 4.1. Two functions:

- `build_vocab(docs)` → a **sorted** list of every distinct token across all documents.
  Sorting matters only so that your column order matches the theory doc's tables
  (`bad, dull, film, good, great, story`).
- `count_matrix(docs, vocab)` → an array of shape `(n_docs, len(vocab))` where cell `[i, j]`
  is the number of times `vocab[j]` occurs in `docs[i]`.

**Recipe:** tokenize each document; for the matrix, make a `Counter` per document and read off
one count per vocabulary word.

In [ ]:
def build_vocab(docs):
    '''Return the sorted list of distinct tokens across all documents.'''
    # TODO 1a
    raise NotImplementedError("TODO 1a: build_vocab")


def count_matrix(docs, vocab):
    '''Return the (n_docs x n_vocab) matrix of raw term counts, as float.'''
    # TODO 1b
    raise NotImplementedError("TODO 1b: count_matrix")

In [ ]:
# CHECK 1 — must match the document-term matrix in theory.md section 4.1
VOCAB = build_vocab(REVIEWS)
assert VOCAB == ["bad", "dull", "film", "good", "great", "story"], VOCAB

X_counts = count_matrix(REVIEWS, VOCAB)
expected = np.array([
    [0, 0, 1, 0, 1, 0],   # D1 great film
    [0, 0, 0, 0, 2, 1],   # D2 great great story
    [0, 0, 1, 1, 0, 0],   # D3 good film
    [1, 0, 1, 0, 0, 0],   # D4 bad film
    [2, 0, 0, 0, 0, 1],   # D5 bad bad story
    [0, 1, 1, 0, 0, 0],   # D6 dull film
], dtype=float)
assert X_counts.shape == (6, 6), X_counts.shape
assert np.array_equal(X_counts, expected), f"\n{X_counts}"
print("✓ CHECK 1 passed — your document-term matrix matches the hand-built table")
print(f"   vocabulary: {VOCAB}")

## Step 2 — TODO 2: IDF, which is Shannon's surprise

Theory 4.4. The **document frequency** `df(t)` is the number of documents containing `t`
**at least once** (not the total number of occurrences — a word said twice in one document
still has df 1 from that document).

$$\text{idf}(t) = \log_2 \frac{N}{\text{df}(t)}$$

We use $\log_2$ so the answer is in **bits**, and reads directly as *the surprise of seeing
this word in a document* — Day 1's very first equation, applied to documents instead of
letters.

Expected results on the review corpus ($N=6$):

| term | df | idf |
|---|---|---|
| film | 4 | 0.585 |
| bad, great, story | 2 | 1.585 |
| good, dull | 1 | 2.585 |

In [ ]:
def compute_idf(X_counts):
    '''Return the length-V vector of idf values, using log2(N / df).

    X_counts: (n_docs x n_vocab) count matrix.
    '''
    # TODO 2
    raise NotImplementedError("TODO 2: compute_idf")

In [ ]:
# CHECK 2 — must match the idf table in theory.md section 4.4
idf = compute_idf(X_counts)
expected_idf = np.array([
    math.log2(6 / 2),   # bad   df=2
    math.log2(6 / 1),   # dull  df=1
    math.log2(6 / 4),   # film  df=4
    math.log2(6 / 1),   # good  df=1
    math.log2(6 / 2),   # great df=2
    math.log2(6 / 2),   # story df=2
])
assert np.allclose(idf, expected_idf), f"\ngot      {idf}\nexpected {expected_idf}"
print("✓ CHECK 2 passed")
for w, v in zip(VOCAB, idf):
    print(f"   {w:6} idf = {v:.3f} bits")
print("\n   'film' is our corpus's version of 'the' — in 4 of 6 documents, so nearly worthless.")
print("   IDF discovered that on its own. No stop-word list required.")

## Step 3 — TODO 3: TF-IDF and L2 normalization

Theory 4.5–4.6. Two steps:

1. **Multiply**: `tfidf[i, j] = counts[i, j] * idf[j]`.
2. **Normalize** each *row* to unit length so a long document doesn't outscore a short one
   just by being long: divide each row by its own L2 norm $\sqrt{\sum_j v_j^2}$.

Guard against a zero-norm row (a document with no known words) so you never divide by zero.

After normalizing, row D1 should be `film = 0.346`, `great = 0.938`.

In [ ]:
def tfidf_matrix(X_counts, idf):
    '''Return the L2-row-normalized tf-idf matrix.'''
    # TODO 3
    raise NotImplementedError("TODO 3: tfidf_matrix")

In [ ]:
# CHECK 3 — must match the normalized table in theory.md section 4.6
X_tfidf = tfidf_matrix(X_counts, idf)
expected_tfidf = np.array([
    [0.000, 0.000, 0.346, 0.000, 0.938, 0.000],
    [0.000, 0.000, 0.000, 0.000, 0.894, 0.447],
    [0.000, 0.000, 0.221, 0.975, 0.000, 0.000],
    [0.938, 0.000, 0.346, 0.000, 0.000, 0.000],
    [0.894, 0.000, 0.000, 0.000, 0.000, 0.447],
    [0.000, 0.975, 0.221, 0.000, 0.000, 0.000],
])
assert np.allclose(X_tfidf, expected_tfidf, atol=1e-3), f"\n{np.round(X_tfidf, 3)}"
assert np.allclose(np.linalg.norm(X_tfidf, axis=1), 1.0), "every row must have length 1"
print("✓ CHECK 3 passed — every document now sits on the unit sphere")
print(np.round(X_tfidf, 3))

### Does the log base matter?

Theory 4.4 claimed that switching from $\log_2$ to $\ln$ scales every tf-idf vector by the
same constant — which **L2 normalization then divides straight back out**. So the normalized
matrices should be *identical*, not merely similar. Let's confirm rather than trust.

In [ ]:
N = len(REVIEWS)
df = (X_counts > 0).sum(axis=0)
idf_ln = np.log(N / df)                      # natural log instead of log2
X_tfidf_ln = tfidf_matrix(X_counts, idf_ln)

print("max absolute difference between the log2 and ln versions:",
      np.abs(X_tfidf - X_tfidf_ln).max())
assert np.allclose(X_tfidf, X_tfidf_ln)
print("✓ identical. The log base is a free choice once you normalize.")

### Reconciling with scikit-learn

`TfidfVectorizer` will *not* reproduce our numbers exactly, and it's worth seeing why rather
than being confused by it later. sklearn computes a **smoothed** IDF:

$$\text{idf}_{\text{sklearn}}(t) = \ln\frac{1 + N}{1 + \text{df}(t)} + 1$$

The `+1`s prevent division by zero for unseen words, and the trailing `+1` stops a word that
appears in *every* document from getting weight exactly zero (which would delete it entirely
rather than merely deprioritize it). Different arithmetic, same idea — and note the *ordering*
of the words by weight is unchanged.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(token_pattern=r"[a-z0-9']+", lowercase=True)
X_sk = vec.fit_transform(REVIEWS).toarray()

print("sklearn vocabulary:", list(vec.get_feature_names_out()))
print(f"\n{'term':8} {'ours (log2)':>12} {'sklearn':>10}")
for w, a, b in zip(VOCAB, idf, vec.idf_):
    print(f"{w:8} {a:12.3f} {b:10.3f}")

print("\nRank order of terms by idf — ours vs sklearn:")
print("  ours   :", [VOCAB[i] for i in np.argsort(-idf)])
print("  sklearn:", [VOCAB[i] for i in np.argsort(-vec.idf_)])
print("\nSame ordering: both agree 'film' is the least informative word.")

## Step 4 — TODO 4: multinomial naive Bayes

Theory 5.4–5.5. Training is **counting and dividing**, exactly like the Day 1 n-gram model.

**Fit.** For each class `c`:
- `log_prior[c] = log(number of docs in c / total docs)`
- Pool all word counts in class `c`. With Laplace smoothing (`alpha=1`) and vocabulary size `V`:

$$P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\left(\sum_{w'} \text{count}(w', c)\right) + \alpha V}$$

  Store `log_prob[c]` = the log of that vector.

**Predict.** Work in logs so hundreds of small probabilities don't underflow to zero:

$$\text{score}(c, d) = \log P(c) + \sum_j x_{dj} \cdot \log P(w_j \mid c)$$

The `x_dj` multiplier is what makes this *multinomial*: a word said twice contributes its
log-probability twice. Then pick the class with the higher score.

In [ ]:
class MultinomialNaiveBayes:
    '''Multinomial naive Bayes with Laplace smoothing, from scratch.'''

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        '''X: (n_docs x n_vocab) counts. y: (n_docs,) class labels 0/1.

        Sets self.classes_, self.log_prior_ (n_classes,), self.log_prob_ (n_classes x n_vocab).
        '''
        # TODO 4a
        raise NotImplementedError("TODO 4a: MultinomialNaiveBayes.fit")

    def joint_log_likelihood(self, X):
        '''Return (n_docs x n_classes): log P(c) + sum_j x_j log P(w_j|c).'''
        # TODO 4b
        raise NotImplementedError("TODO 4b: joint_log_likelihood")

    def predict(self, X):
        return self.classes_[np.argmax(self.joint_log_likelihood(X), axis=1)]

In [ ]:
# CHECK 4 — must match the hand-worked model in theory.md section 5.5
nb = MultinomialNaiveBayes(alpha=1.0).fit(X_counts, LABELS)

# priors: 3 positive, 3 negative
assert np.allclose(np.exp(nb.log_prior_), [0.5, 0.5]), np.exp(nb.log_prior_)

# every likelihood is a multiple of 1/13  (class total 7 + V 6)
probs = np.exp(nb.log_prob_)
neg_expected = np.array([4, 2, 3, 1, 1, 2]) / 13   # bad dull film good great story
pos_expected = np.array([1, 1, 3, 2, 4, 2]) / 13
assert np.allclose(probs[0], neg_expected), probs[0] * 13
assert np.allclose(probs[1], pos_expected), probs[1] * 13
assert np.allclose(probs.sum(axis=1), 1.0), "each class's word probabilities must sum to 1"

# the worked example: "great story" -> positive at exactly 4:1 odds
test = count_matrix(["great story"], VOCAB)
jll = nb.joint_log_likelihood(test)[0]
p_pos, p_neg = math.exp(jll[1]), math.exp(jll[0])
assert abs(p_pos - 4 / 169) < 1e-12, p_pos
assert abs(p_neg - 1 / 169) < 1e-12, p_neg
assert nb.predict(test)[0] == 1

print("✓ CHECK 4 passed")
print(f"   P(pos) x P(great|pos) x P(story|pos) = 4/169 = {p_pos:.5f}")
print(f"   P(neg) x P(great|neg) x P(story|neg) = 1/169 = {p_neg:.5f}")
print(f"   -> POSITIVE, at odds of exactly {p_pos / p_neg:.0f} to 1")
print("\n   'story' contributed 2/13 to BOTH sides and cancelled out entirely.")
print("   The whole decision was made by 'great'. Neutral words cancel; discriminative words decide.")

### Watch it break #1: double counting

The independence assumption says every word is a fresh, independent piece of evidence. So
saying `bad` three times counts as *three separate confirmations* and the odds get cubed.

Is that right? A human reads "bad bad bad" as **one** person with a limited vocabulary, not as
three independent witnesses. Naive Bayes cannot tell the difference — and this is exactly why
binarizing counts (the Bernoulli model, theory 5.6) often works better on long documents.

In [ ]:
for text in ["bad", "bad bad", "bad bad bad"]:
    jll = nb.joint_log_likelihood(count_matrix([text], VOCAB))[0]
    odds = math.exp(jll[0] - jll[1])
    print(f"{text:15} -> negative at {odds:8.1f} : 1 odds")

print("\nThe odds are 4, 16, 64 — a clean power of 4. Each repetition multiplies the")
print("evidence again, because the model believes each word arrived independently.")

## Step 5 — TODO 5: logistic regression

Theory 6.1–6.3. Four small pieces:

1. **`sigmoid(z)`** = $1/(1+e^{-z})$. Use `np.clip(z, -500, 500)` first so `np.exp` doesn't overflow.
2. **`predict_proba(X, w, b)`** = `sigmoid(X @ w + b)`.
3. **`log_loss(y, yhat)`** = $-\frac{1}{m}\sum [\,y\log\hat y + (1-y)\log(1-\hat y)\,]$ — Day 1's
   cross-entropy. Clip `yhat` into `[1e-12, 1-1e-12]` so `log(0)` never happens.
4. **`gradient_step(X, y, w, b, lr)`** → new `(w, b)` using
   $\frac{\partial L}{\partial w_j} = \frac{1}{m}\sum_i(\hat y_i - y_i)x_{ij}$ and
   $\frac{\partial L}{\partial b} = \frac{1}{m}\sum_i(\hat y_i - y_i)$.

Read that gradient in English: **(prediction − truth) × (feature value), averaged.**

In [ ]:
def sigmoid(z):
    '''Logistic squashing function, overflow-safe.'''
    # TODO 5a
    raise NotImplementedError("TODO 5a: sigmoid")


def predict_proba(X, w, b):
    '''P(y=1 | x) for every row of X.'''
    # TODO 5b
    raise NotImplementedError("TODO 5b: predict_proba")


def log_loss(y, yhat):
    '''Mean binary cross-entropy. This is Day 1's cross-entropy with two outcomes.'''
    # TODO 5c
    raise NotImplementedError("TODO 5c: log_loss")


def gradient_step(X, y, w, b, lr):
    '''One step of gradient descent. Returns the updated (w, b).'''
    # TODO 5d
    raise NotImplementedError("TODO 5d: gradient_step")

In [ ]:
# CHECK 5 — one gradient step must match the hand-worked table in theory.md section 6.3
assert abs(sigmoid(0) - 0.5) < 1e-12
assert abs(sigmoid(2) - 0.8807970779778823) < 1e-12
assert abs(sigmoid(-2) + sigmoid(2) - 1.0) < 1e-12          # symmetry

w0, b0 = np.zeros(len(VOCAB)), 0.0
yhat0 = predict_proba(X_counts, w0, b0)
assert np.allclose(yhat0, 0.5), "with zero weights every document must score exactly 0.5"
assert abs(log_loss(LABELS, yhat0) - math.log(2)) < 1e-12, "loss must start at ln 2 = 0.693"

w1, b1 = gradient_step(X_counts, LABELS, w0, b0, lr=1.0)
expected_w = np.array([-3, -1, 0, 1, 3, 0]) / 12   # bad dull film good great story
assert np.allclose(w1, expected_w), f"\ngot      {w1}\nexpected {expected_w}"
assert abs(b1) < 1e-12, "balanced classes -> the bias stays at 0"
assert abs(log_loss(LABELS, predict_proba(X_counts, w1, b1)) - 0.567455) < 1e-5

print("✓ CHECK 5 passed — after ONE step, from all-zero weights:")
for word, weight in zip(VOCAB, w1):
    bar = "+" * int(abs(weight) * 40) if weight > 0 else "-" * int(abs(weight) * 40)
    print(f"   {word:6} {weight:+.3f}  {bar}")
print(f"\n   loss: {math.log(2):.6f} -> {log_loss(LABELS, predict_proba(X_counts, w1, b1)):.6f}")
print("\n   'film' and 'story' got EXACTLY zero — they appear equally in both classes,")
print("   so they carry no signal about the boundary. Naive Bayes still gave 'film' 3/13")
print("   in both classes; logistic regression simply refuses to spend weight on it.")
print("   Naive Bayes models the data. Logistic regression models the boundary.")

In [ ]:
def train_logreg(X, y, lr=0.5, epochs=2000, l2=0.0, verbose=False):
    '''Full-batch gradient descent. Provided for you — it just loops gradient_step.'''
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for epoch in range(epochs):
        w, b = gradient_step(X, y, w, b, lr)
        if l2:                                  # simple L2 weight decay
            w -= lr * l2 * w
        loss = log_loss(y, predict_proba(X, w, b))
        history.append(loss)
        if verbose and epoch % 500 == 0:
            print(f"  epoch {epoch:5d}  loss {loss:.5f}")
    return w, b, history


w, b, hist = train_logreg(X_tfidf, LABELS, lr=1.0, epochs=3000, verbose=True)
print(f"\nfinal loss: {hist[-1]:.6f}\n")
print("learned weights on the tf-idf features:")
for word, weight in sorted(zip(VOCAB, w), key=lambda t: -t[1]):
    print(f"   {word:6} {weight:+.3f}")
print("\nPositive words on top, negative on the bottom, and the two neutral words")
print("('film', 'story') pinned near zero. It rediscovered the sentiment axis from 6 examples.")

## Step 6 — TODO 6: precision, recall, F1

Theory 8.2–8.4. From predictions and truth, count the four confusion-matrix cells and derive
the three metrics:

$$\text{Precision} = \frac{TP}{TP+FP} \qquad \text{Recall} = \frac{TP}{TP+FN} \qquad F_1 = \frac{2PR}{P+R}$$

Return 0.0 rather than a `NaN` when a denominator is zero (a model that flags nothing has
undefined precision; scoring it 0 is the convention).

In [ ]:
def confusion_counts(y_true, y_pred):
    '''Return (tp, fp, fn, tn) treating class 1 as 'positive'.'''
    # TODO 6a
    raise NotImplementedError("TODO 6a: confusion_counts")


def precision_recall_f1(y_true, y_pred):
    '''Return (precision, recall, f1).'''
    # TODO 6b
    raise NotImplementedError("TODO 6b: precision_recall_f1")

In [ ]:
# CHECK 6 — the worked example from theory.md section 8.3
y_true = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 1, 0, 1, 1, 0, 0, 0])

tp, fp, fn, tn = confusion_counts(y_true, y_pred)
assert (tp, fp, fn, tn) == (4, 2, 1, 3), (tp, fp, fn, tn)

p, r, f1 = precision_recall_f1(y_true, y_pred)
assert abs(p - 2 / 3) < 1e-12, p
assert abs(r - 4 / 5) < 1e-12, r
assert abs(f1 - 8 / 11) < 1e-12, f1

# cross-check against sklearn
from sklearn.metrics import precision_score, recall_score, f1_score
assert abs(p - precision_score(y_true, y_pred)) < 1e-12
assert abs(r - recall_score(y_true, y_pred)) < 1e-12
assert abs(f1 - f1_score(y_true, y_pred)) < 1e-12

print("✓ CHECK 6 passed — and matches sklearn exactly")
print(f"   TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"   precision = 4/6 = {p:.4f}   (a third of its flags are wrong)")
print(f"   recall    = 4/5 = {r:.4f}   (it catches 80% of the positives)")
print(f"   F1        = 8/11 = {f1:.4f}")
print(f"   accuracy  = 7/10 = 0.7000   <- tells you neither of the above")

### Why the *harmonic* mean?

Because it refuses to be fooled. Consider the useless classifier that flags **everything** as
positive on a dataset where only 1% is positive. Its recall is a perfect 1.00, its precision is
0.01. The ordinary average calls that a respectable coin-flip model. The harmonic mean does not.

In [ ]:
prec, rec = 0.01, 1.00
arithmetic = (prec + rec) / 2
harmonic = 2 * prec * rec / (prec + rec)
print(f"precision {prec:.2f}, recall {rec:.2f}")
print(f"   arithmetic mean : {arithmetic:.4f}   <- looks fine!")
print(f"   harmonic mean(F1): {harmonic:.4f}   <- correctly damning")

print("\nF1 across the whole precision range at recall = 1.0:")
print(f"{'precision':>10} {'arithmetic':>12} {'F1':>8}")
for prec in [0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
    print(f"{prec:10.2f} {(prec + 1) / 2:12.3f} {2 * prec / (prec + 1):8.3f}")
print("\nThe harmonic mean is always dragged toward the SMALLER of the two numbers.")
print("You cannot buy a good F1 by maxing one metric and abandoning the other.")

---

# Part 2 — 50,000 real movie reviews

The toy corpus proved the code is correct. Now the interesting question: **what happens at
scale?** We use the IMDb dataset (Maas et al., 2011): 25,000 labeled reviews for training and
25,000 for testing, split exactly 50/50 positive and negative.

From here on we use scikit-learn's implementations, so this half runs whether or not you
finished the TODOs — but we'll validate your from-scratch naive Bayes against sklearn's first.

In [ ]:
def load_imdb():
    '''Load IMDb. Tries HuggingFace datasets, falls back to the Stanford tarball.'''
    try:
        from datasets import load_dataset
        ds = load_dataset("imdb")
        tr, te = ds["train"].shuffle(seed=42), ds["test"].shuffle(seed=42)
        return list(tr["text"]), np.array(tr["label"]), list(te["text"]), np.array(te["label"])
    except Exception as e:
        print(f"datasets unavailable ({e}); downloading the tarball instead...")

    import tarfile, urllib.request, pathlib, random
    url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    path = pathlib.Path("aclImdb_v1.tar.gz")
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    if not pathlib.Path("aclImdb").exists():
        with tarfile.open(path) as tar:
            tar.extractall(".")

    def read_split(split):
        texts, labels = [], []
        for label, name in [(1, "pos"), (0, "neg")]:
            for f in sorted(pathlib.Path(f"aclImdb/{split}/{name}").glob("*.txt")):
                texts.append(f.read_text(encoding="utf-8"))
                labels.append(label)
        idx = list(range(len(texts)))
        random.Random(42).shuffle(idx)
        return [texts[i] for i in idx], np.array([labels[i] for i in idx])

    Xtr, ytr = read_split("train")
    Xte, yte = read_split("test")
    return Xtr, ytr, Xte, yte


X_train_text, y_train, X_test_text, y_test = load_imdb()
print(f"train: {len(X_train_text):,} reviews   positive rate {y_train.mean():.3f}")
print(f"test : {len(X_test_text):,} reviews   positive rate {y_test.mean():.3f}")
print(f"average review length: {np.mean([len(t.split()) for t in X_train_text]):.0f} words")
print("\n--- a sample review ---")
print(X_train_text[0][:400], "...")
print(f"\nlabel: {'POSITIVE' if y_train[0] else 'NEGATIVE'}")

## The matrix is enormous, and almost entirely empty

Theory 4.1 warned about this. Let's measure it: one column per distinct word in 25,000 reviews,
and each review touching a tiny handful of them.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = TfidfVectorizer(use_idf=False, norm=None)   # raw counts
Xtr_counts = CountVectorizer().fit(X_train_text)
Xtr_c = Xtr_counts.transform(X_train_text)
Xte_c = Xtr_counts.transform(X_test_text)

n_docs, n_feats = Xtr_c.shape
print(f"document-term matrix: {n_docs:,} x {n_feats:,} = {n_docs * n_feats:,} cells")
print(f"non-zero cells      : {Xtr_c.nnz:,}")
print(f"density             : {100 * Xtr_c.nnz / (n_docs * n_feats):.3f}%  <- 99.8% zeros")
print(f"distinct words per review: {Xtr_c.nnz / n_docs:.0f} on average")
print(f"\nDense float64 would need {n_docs * n_feats * 8 / 1e9:.1f} GB.")
print(f"Sparse storage uses about {Xtr_c.data.nbytes / 1e6:.0f} MB.")

### Does IDF actually find the stop words?

Theory 4.4 claimed IDF automatically discovers what a hand-written stop-word list encodes.
Here are the fifteen lowest-IDF words in 25,000 movie reviews — nobody told the model these
were function words.

In [ ]:
tfidf_vec = TfidfVectorizer()
Xtr_t = tfidf_vec.fit_transform(X_train_text)      # NOTE: fit on TRAIN ONLY
Xte_t = tfidf_vec.transform(X_test_text)

features = np.array(tfidf_vec.get_feature_names_out())
order = np.argsort(tfidf_vec.idf_)
print("LOWEST idf (least informative):")
print("  ", ", ".join(f"{features[i]} ({tfidf_vec.idf_[i]:.2f})" for i in order[:15]))
print("\nHIGHEST idf (appear in a single review):")
print("  ", ", ".join(features[i] for i in order[-8:]))
print(f"\nidf range: {tfidf_vec.idf_.min():.2f} to {tfidf_vec.idf_.max():.2f}")

## Does our from-scratch naive Bayes match sklearn's?

The real test of the code in Part 1. Same data, same smoothing — the predictions should agree
essentially perfectly. (Skip this cell if you haven't done TODO 4 yet.)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

try:
    ours = MultinomialNaiveBayes(alpha=1.0).fit(Xtr_c.toarray()[:5000], y_train[:5000])
    theirs = MultinomialNB(alpha=1.0).fit(Xtr_c[:5000], y_train[:5000])
    ours_pred = ours.predict(Xte_c.toarray()[:2000])
    theirs_pred = theirs.predict(Xte_c[:2000])
    agree = (ours_pred == theirs_pred).mean()
    print(f"agreement between your naive Bayes and sklearn's: {100 * agree:.2f}%")
    print("✓ your from-scratch implementation is the real thing" if agree > 0.99 else "✗ mismatch")
except NotImplementedError:
    print("(TODO 4 not implemented yet — skipping the cross-check)")

## The main experiment

Every combination that matters, scored on the held-out 25,000 test reviews. This is the day's
deliverable: **TF-IDF + logistic regression, evaluated with precision, recall and F1.**

In [ ]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix

bin_vec = CountVectorizer(binary=True)
Xtr_b, Xte_b = bin_vec.fit_transform(X_train_text), bin_vec.transform(X_test_text)

bigram_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
Xtr_t2, Xte_t2 = bigram_vec.fit_transform(X_train_text), bigram_vec.transform(X_test_text)

experiments = [
    ("Naive Bayes (counts)",         MultinomialNB(),                    Xtr_c,  Xte_c),
    ("Naive Bayes (binary/Bernoulli)", BernoulliNB(),                    Xtr_b,  Xte_b),
    ("Naive Bayes (tf-idf)",         MultinomialNB(),                    Xtr_t,  Xte_t),
    ("Logistic regression (counts)", LogisticRegression(max_iter=2000),  Xtr_c,  Xte_c),
    ("Logistic regression (tf-idf)", LogisticRegression(max_iter=2000),  Xtr_t,  Xte_t),
    ("Linear SVM (tf-idf)",          LinearSVC(C=0.1),                   Xtr_t,  Xte_t),
    ("LogReg (tf-idf + bigrams)",    LogisticRegression(max_iter=2000),  Xtr_t2, Xte_t2),
]

print(f"{'model':34} {'acc':>7} {'prec':>7} {'recall':>7} {'F1':>7}")
print("-" * 68)
fitted = {}
for name, model, Xa, Xb in experiments:
    model.fit(Xa, y_train)
    pred = model.predict(Xb)
    fitted[name] = (model, Xb, pred)
    p, r, f1 = precision_score(y_test, pred), recall_score(y_test, pred), f1_score(y_test, pred)
    print(f"{name:34} {accuracy_score(y_test, pred):7.4f} {p:7.4f} {r:7.4f} {f1:7.4f}")

Three things to read off that table:

1. **TF-IDF is worth roughly 1.6 accuracy points** to logistic regression over raw counts —
   real, but more modest than TF-IDF's reputation suggests.
2. **Logistic regression and the linear SVM are indistinguishable.** Once the representation is
   good, the specific linear classifier barely matters — which is exactly Joachims' point
   (theory 7): text in a bag-of-words space is already nearly linearly separable, so all these
   models are arguing over *which* separating plane to pick, not whether one exists.
3. **Naive Bayes trails by about 7 points** — and Bernoulli beats multinomial-on-counts, because
   IMDb reviews are long and repeated words inflate multinomial's evidence.

In [ ]:
model, Xb, pred = fitted["Logistic regression (tf-idf)"]
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print("Confusion matrix for TF-IDF + logistic regression:\n")
print(f"{'':22} {'predicted pos':>14} {'predicted neg':>14}")
print(f"{'actually positive':22} {tp:>14,} {fn:>14,}")
print(f"{'actually negative':22} {fp:>14,} {tn:>14,}")
print(f"\nprecision = {tp}/({tp}+{fp}) = {tp / (tp + fp):.4f}")
print(f"recall    = {tp}/({tp}+{fn}) = {tp / (tp + fn):.4f}")
print(f"\nThe {fp:,} false positives and {fn:,} false negatives are almost perfectly balanced —")
print("which is what you'd expect from a 50/50 dataset with a 0.5 decision threshold.")

## Regularization: watching a model overfit in real time

Theory 6.4. `C` is the inverse regularization strength, so **small C = strong regularization**.
Watch the train and test columns pull apart.

In [ ]:
print(f"{'C':>8} {'train acc':>11} {'test acc':>10} {'test F1':>9}   gap")
print("-" * 52)
for C in [0.01, 0.1, 1, 10, 100]:
    m = LogisticRegression(max_iter=3000, C=C).fit(Xtr_t, y_train)
    tr = accuracy_score(y_train, m.predict(Xtr_t))
    te = accuracy_score(y_test, m.predict(Xte_t))
    f1 = f1_score(y_test, m.predict(Xte_t))
    print(f"{C:>8} {tr:>11.4f} {te:>10.4f} {f1:>9.4f}   {tr - te:+.4f}")

print("\nAt C=100 the model classifies its own training data PERFECTLY (100.0%) and does")
print("worse on new reviews than the far humbler C=1. That growing gap between the train")
print("and test columns is overfitting, visible in one table.")
print("At C=0.01 it underfits — too much regularization and it can't learn enough.")
print("The best value is in the middle, and you find it by measuring, never by intuition.")

## The crossover: Ng & Jordan (2001) on real data

Theory 6.5 predicted that **naive Bayes should win with little data and logistic regression
should win with a lot** — because naive Bayes converges faster but has a higher error floor.

We refit the vectorizer on each subset, so no information leaks in from the full corpus.
This cell takes a minute or so.

In [ ]:
rng = np.random.RandomState(0)
pos_idx = rng.permutation(np.where(y_train == 1)[0])
neg_idx = rng.permutation(np.where(y_train == 0)[0])

print(f"{'n docs':>8} {'vocab':>8} {'NB acc':>9} {'LR acc':>9}   winner")
print("-" * 50)
curve = []
for n in [20, 50, 200, 500, 1000, 5000, 25000]:
    idx = np.concatenate([pos_idx[: n // 2], neg_idx[: n // 2]])
    sub = [X_train_text[i] for i in idx]
    ys = y_train[idx]

    cv = CountVectorizer().fit(sub)        # fit on the subset ONLY
    tv = TfidfVectorizer().fit(sub)
    nb_acc = accuracy_score(y_test, MultinomialNB().fit(cv.transform(sub), ys).predict(cv.transform(X_test_text)))
    lr_acc = accuracy_score(y_test, LogisticRegression(max_iter=3000).fit(tv.transform(sub), ys).predict(tv.transform(X_test_text)))
    curve.append((n, nb_acc, lr_acc))
    win = "NB" if nb_acc > lr_acc else "LR"
    print(f"{n:>8,} {len(cv.vocabulary_):>8,} {nb_acc:>9.4f} {lr_acc:>9.4f}   {win}")

print("\nThe crossover lands right around 1,000 documents, exactly as the theory predicts.")
print("Naive Bayes plateaus hard: it gains only ~2.6 points between 1k and 25k documents,")
print("because its ceiling is set by the independence assumption, not by how much data it has.")

In [ ]:
import matplotlib.pyplot as plt

ns = [c[0] for c in curve]
plt.figure(figsize=(7, 4.5))
plt.semilogx(ns, [c[1] for c in curve], "o-", label="Naive Bayes")
plt.semilogx(ns, [c[2] for c in curve], "s-", label="Logistic regression")
plt.xlabel("training documents")
plt.ylabel("test accuracy")
plt.title("Ng & Jordan (2001), reproduced on IMDb")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## What did each model actually learn?

A revealing comparison. For logistic regression the weight *is* the evidence. For naive Bayes
the equivalent is the log-ratio $\log P(w|\text{pos}) - \log P(w|\text{neg})$.

Watch what happens when we ask each one for its twenty most positive words — and check how
many documents each of those words actually appears in.

In [ ]:
lr_model = fitted["Logistic regression (tf-idf)"][0]
coef = lr_model.coef_[0]
doc_freq_t = np.asarray((Xtr_t > 0).sum(axis=0)).ravel()

nb_model = MultinomialNB().fit(Xtr_c, y_train)
feat_c = np.array(Xtr_counts.get_feature_names_out())
log_ratio = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]
doc_freq_c = np.asarray((Xtr_c > 0).sum(axis=0)).ravel()

print("LOGISTIC REGRESSION — top positive words")
for i in np.argsort(coef)[-12:][::-1]:
    print(f"   {features[i]:14} weight {coef[i]:+.2f}   appears in {doc_freq_t[i]:>6,} reviews")

print("\nNAIVE BAYES — top positive words")
for i in np.argsort(log_ratio)[-12:][::-1]:
    print(f"   {feat_c[i]:14} ratio  {log_ratio[i]:+.2f}   appears in {doc_freq_c[i]:>6,} reviews")

In [ ]:
print("LOGISTIC REGRESSION — top negative words")
print("  ", ", ".join(features[i] for i in np.argsort(coef)[:15]))

Look at the document counts. Logistic regression's evidence rests on words appearing in
**hundreds to thousands** of reviews — `great`, `excellent`, `perfect`, `wonderful`. Naive
Bayes' most confident positive evidence is a list of **proper nouns appearing in 9 to 40
reviews out of 25,000**: character names from a handful of well-liked films that happen never
to appear in the negative class, so smoothing hands them an extreme ratio.

That is theory 6.4 in action. **Regularization** asks of every weight, "is this earning its
size?", and shrinks the ones resting on thin evidence. Naive Bayes has no equivalent mechanism
— smoothing prevents zeros, but nothing prevents extremes.

(One entertaining detail in logistic regression's negative list: **`script`**. Not a negative
word in isolation — but reviewers who mention "the script" are usually complaining about it.)

## Watch it break #2: naive Bayes double-counts

Theory 5.7. A surgical experiment. We **duplicate every feature column**, so each word appears
twice, perfectly correlated. Not one bit of new information has been added.

A model that understands correlation should be unmoved. A model that assumes independence
should treat the copy as a second, independent witness.

In [ ]:
from scipy.sparse import hstack

Xtr_dup = hstack([Xtr_c, Xtr_c]).tocsr()
Xte_dup = hstack([Xte_c, Xte_c]).tocsr()

nb_o = MultinomialNB().fit(Xtr_c, y_train)
nb_d = MultinomialNB().fit(Xtr_dup, y_train)
lr_o = LogisticRegression(max_iter=2000).fit(Xtr_c, y_train)
lr_d = LogisticRegression(max_iter=2000).fit(Xtr_dup, y_train)

nb_lo = np.abs(np.diff(nb_o.predict_log_proba(Xte_c), axis=1)).mean()
nb_ld = np.abs(np.diff(nb_d.predict_log_proba(Xte_dup), axis=1)).mean()
lr_mo = np.abs(lr_o.decision_function(Xte_c)).mean()
lr_md = np.abs(lr_d.decision_function(Xte_dup)).mean()

print(f"{'':28} {'original':>10} {'duplicated':>12}   ratio")
print("-" * 64)
print(f"{'NB mean |log-odds|':28} {nb_lo:>10.2f} {nb_ld:>12.2f}   {nb_ld / nb_lo:.2f}x")
print(f"{'LR mean |margin|':28} {lr_mo:>10.2f} {lr_md:>12.2f}   {lr_md / lr_mo:.2f}x")
print(f"\n{'NB F1':28} {f1_score(y_test, nb_o.predict(Xte_c)):>10.4f} {f1_score(y_test, nb_d.predict(Xte_dup)):>12.4f}")
print(f"{'LR F1':28} {f1_score(y_test, lr_o.predict(Xte_c)):>10.4f} {f1_score(y_test, lr_d.predict(Xte_dup)):>12.4f}")

print("\nNaive Bayes' confidence doubles EXACTLY. It cannot tell that the second copy of")
print("each word carries no new information — the independence assumption forbids it.")
print("Logistic regression just splits each weight between the two copies and carries on.")
print("\nNote the F1 columns barely move: naive Bayes' *decisions* survive, only its")
print("*probabilities* are nonsense. Badly calibrated, still a decent decision-maker.")

## Watch it break #3: a bag of words cannot read

The deepest limitation, and it belongs to the *representation*, not to either classifier.

In [ ]:
a = tfidf_vec.transform(["the movie was good, not bad at all"])
b = tfidf_vec.transform(["the movie was bad, not good at all"])

print("Are the two vectors identical?", (a != b).nnz == 0)
print("\nThose sentences mean opposite things. To a bag of words they are the same point")
print("in space — so ANY model built on this representation must give them the same")
print("answer. No amount of training data can fix it. Only a better representation can.")

In [ ]:
probes = [
    "the movie was good, not bad at all",
    "the movie was bad, not good at all",
    "i expected this to be terrible but it was wonderful",
    "i expected this to be wonderful but it was terrible",
    "not a single boring moment in this masterpiece",
    "this is not a masterpiece, it is a single boring moment",
]

lr_uni = fitted["Logistic regression (tf-idf)"][0]
lr_bi = fitted["LogReg (tf-idf + bigrams)"][0]
p_uni = lr_uni.predict_proba(tfidf_vec.transform(probes))[:, 1]
p_bi = lr_bi.predict_proba(bigram_vec.transform(probes))[:, 1]

print(f"{'P(pos) unigram':>15} {'+ bigrams':>11}   sentence")
print("-" * 78)
for text, pu, pb in zip(probes, p_uni, p_bi):
    print(f"{pu:>15.3f} {pb:>11.3f}   {text}")

print("\nThe first two sentences get IDENTICAL unigram probabilities — they must, the")
print("vectors are identical. Bigrams pull them apart (0.087 vs 0.024) by turning")
print("'not bad' and 'not good' into features of their own.")
print("\nBut the patch doesn't scale: catching 'not' five words away needs 5-grams,")
print("and the feature count explodes. Genuinely fixing this takes models that read")
print("in order — RNNs on Day 7, attention on Day 10.")

## The precision/recall dial

Theory 8.3. **Nothing about the model changes below.** Same weights, same training. Only the
threshold at which a probability becomes a decision moves — and precision and recall trade off
against each other along the whole range.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

prob = lr_uni.predict_proba(Xte_t)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, prob)

print(f"{'threshold':>10} {'precision':>11} {'recall':>9}")
print("-" * 33)
for target in [0.50, 0.70, 0.90, 0.95, 0.99]:
    i = int(np.argmin(np.abs(prec - target)))
    t = thr[min(i, len(thr) - 1)]
    print(f"{t:>10.3f} {prec[i]:>11.3f} {rec[i]:>9.3f}")

print(f"\naverage precision (area under the PR curve): {average_precision_score(y_test, prob):.4f}")
print("\nTo make this classifier 99% precise you must accept that it finds only ~26% of")
print("the positive reviews. Which point you pick is a business decision, not a modeling")
print("one — which is why a single accuracy number for a deployed classifier means little.")

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.plot(rec, prec, lw=2)
plt.axhline(y_test.mean(), ls="--", c="gray", lw=1, label=f"chance ({y_test.mean():.2f})")
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("Precision-recall trade-off: TF-IDF + logistic regression")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## The most common bug in text classification: leakage

Theory 8.5, rule 2. If you call `fit_transform` on your **whole** dataset before splitting,
the vocabulary and the IDF values are computed using the test documents. The model gets
information from data it is supposed to have never seen, and your reported score is a lie.

Here's the size of the hole.

In [ ]:
leaky_vec = TfidfVectorizer().fit(X_train_text + X_test_text)   # WRONG on purpose

train_only = set(tfidf_vec.vocabulary_)
combined = set(leaky_vec.vocabulary_)
unseen = combined - train_only

print(f"vocabulary from train only     : {len(train_only):,}")
print(f"vocabulary from train + test   : {len(combined):,}")
print(f"words only the test set knows  : {len(unseen):,}  ({100 * len(unseen) / len(combined):.1f}%)")
print("\nOver a quarter of the combined vocabulary is words the model has no business")
print("knowing about. Fit the vectorizer on training data ONLY, then `.transform()` the")
print("test set — which is exactly what every cell above did.")

## What you just proved

1. **IDF is Shannon's surprise.** `log(N/df)` is `log(1/p)` — Day 1's very first equation,
   with "this word appears in a document" as the event. It rediscovered the stop-word list
   from data, with no list.

2. **Training naive Bayes is counting and dividing** — the third time in three days — and it
   breaks on zeros in exactly the way the Day 1 n-gram model did, and gets fixed by exactly
   the same Laplace smoothing.

3. **Naive Bayes models the data; logistic regression models the boundary.** After one gradient
   step, logistic regression had already assigned the neutral words `film` and `story` a weight
   of exactly zero. Naive Bayes spent probability mass on them because its job is to describe
   how documents are generated.

4. **The generative/discriminative crossover is real and it's around 1,000 documents.**
   Below that, use naive Bayes. Above it, use logistic regression.

5. **Regularization is what separates a model from a memorizer.** At C=100 our classifier
   scored 100.0% on training data and got *worse* on real reviews.

6. **The loss we minimized is Day 1's cross-entropy**, and the sigmoid we squashed through
   is the two-class version of the softmax that sits at the output of every Transformer.
   We built the last layer of GPT today.

7. **A bag of words cannot read.** "good, not bad" and "bad, not good" are the same vector,
   and therefore get the same prediction, forever. That is the wall — and Day 4's word
   embeddings are the first serious attempt to climb it.

### Optional stretch: multi-class

Everything above was binary. Swap in a 20-class dataset and almost nothing changes — except
that you now have to decide how to *average* precision and recall across classes (theory 8.5),
and macro vs micro give genuinely different answers on imbalanced data.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import classification_report

train20 = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
test20 = fetch_20newsgroups(subset="test", remove=("headers", "footers", "quotes"))

v20 = TfidfVectorizer(min_df=2)
A, B = v20.fit_transform(train20.data), v20.transform(test20.data)

for name, clf in [("Naive Bayes", MultinomialNB(alpha=0.05)),
                  ("Logistic regression", LogisticRegression(max_iter=3000, C=10))]:
    pred = clf.fit(A, train20.target).predict(B)
    print(f"{name:22} accuracy {accuracy_score(test20.target, pred):.4f}   "
          f"macro-F1 {f1_score(test20.target, pred, average='macro'):.4f}   "
          f"micro-F1 {f1_score(test20.target, pred, average='micro'):.4f}")

print("\nmicro-F1 equals accuracy for single-label problems — every DOCUMENT counts equally.")
print("macro-F1 averages the per-class F1s — every CLASS counts equally, so the small")
print("classes the model handles badly drag it down. Always say which one you're reporting.")

---

**Next:** Day 4 — word embeddings. Every classifier today treated `excellent` and `superb` as
two unrelated columns, learning nothing about one from the other. We're about to fix that, and
it changes everything.